# Tutorial 5c - Tensor orientations and spin systems for simulation



```
      _
    /|_|\   
   / / \ \  
  /_/   \_\  
  \ \   / /  
   \ \_/ /  
    \|_|/  

```
SOPRANO: a Python library for generation, manipulation and analysis of large batches of crystalline structures


*Developed within the CCP-NC project. Copyright STFC 2022–2026*

This tutorial shows how to extract the orientation of each NMR tensor (magnetic shielding, electric field gradient, dipolar coupling, and J coupling) from a `.magres` file, select which spins to simulate, and export them as a spin system for [SIMPSON](https://inano.au.dk/about/research-centers-and-projects/nmr/software/simpson) or [MRSimulator](https://mrsimulator.readthedocs.io/en/latest/introduction/getting_started.html). The emphasis is on tensor orientation: a multi-spin or finite-speed MAS lineshape depends on how the principal axes of interacting tensors are oriented relative to one another, which principal values alone do not capture.

This tutorial focuses on preparing and verifying the spin system rather than writing full simulation pulse scripts (Section 9 links to external simulation guides). Loading `.magres` tensors and principal values is covered in [Tutorial 5a](./05a-nmr-parameters.ipynb), which you should read first. [Tutorial 5b](./05b-nmr-2d-spectra.ipynb) covers 2D correlation maps and is not required here.

Prerequisites: [Tutorial 5a](./05a-nmr-parameters.ipynb) and basic familiarity with `ase.Atoms` and NumPy. Sections marked Advanced are skippable on a first read.

**Contents / suggested route**

- 1 - Why orientation matters
- 2 - The NMRTensor class
  - 2.1 - Building a tensor from a matrix
  - 2.2 - Eigenvalue ordering conventions
- 3 - Euler angle conventions
  - 3.1 - ZYZ vs ZXZ, active vs passive
  - 3.2 - Equivalent Euler angle sets
- 4 - Orientations from .magres data
  - 4.1 - Shielding (MSEuler)
  - 4.2 - EFG (EFGEuler)
  - 4.3 - Exporting angles to SIMPSON
  - 4.4 - Relative orientations
  - 4.5 - Dipolar tensor orientation
- 5 - Selecting the spins to simulate
  - 5.1 - By element, label, or index
  - 5.2 - By distance
- 6 - Building a spin system
  - 6.1 - Sites
  - 6.2 - Dipolar couplings
  - 6.3 - J couplings
  - 6.4 - SpinSystem
- 7 - Exporting the spin system
  - 7.1 - SIMPSON
  - 7.2 - MRSimulator
- 8 - Advanced: quadrupole cross-terms
- 9 - Summary and next steps

In [1]:
import json
import os

import numpy as np
import pandas as pd
import ipywidgets as widgets
from ase import Atoms
from ase import io as ase_io
from scipy.spatial.transform import Rotation as R
from weas_widget import WeasWidget

from soprano.nmr import NMRTensor, MagneticShielding, Site
from soprano.nmr.coupling import DipolarCoupling, JCoupling
from soprano.properties.linkage import Molecules
from soprano.properties.nmr import (
    DipolarEuler,
    DipolarTensor,
    EFGEuler,
    EFGTensor,
    MSEuler,
    MSTensor,
    get_dipolar_couplings,
    get_j_couplings,
    get_sites,
    get_spin_system,
)
from soprano.scripts.cli_utils import reload_as_molecular_crystal
from soprano.selection import AtomSelection

## 1 - Why orientation matters

Each anisotropic NMR interaction is described by a $3 \times 3$ tensor: its principal values determine the strength of the interaction, and its principal axes determine how it points relative to the molecule or crystal. In a powder spectrum, the lineshape depends not only on the principal values, but also on the relative orientations of the interacting tensors (and, under MAS, their orientation relative to the rotor axis).

Take a ¹H–¹³C spin pair. The dipolar coupling is an axially symmetric tensor whose unique axis lies along the C–H bond, while the ¹³C shielding tensor has its own principal axes set by local bonding. Rotating those two tensors relative to each other changes the resulting powder pattern even when all principal values remain identical.

Tutorial 5a worked with principal values and infinite-speed MAS. Finite-speed MAS and full multi-spin lineshapes require a spin-dynamics simulator, where every interaction carries three Euler angles $(\alpha, \beta, \gamma)$ placing its principal-axis system (PAS) in a common crystal/molecular frame. Getting those angles—and each program's conventions—right is the focus of this tutorial.

## 2 - The NMRTensor class

`NMRTensor` is the general class behind every tensor in Soprano. It takes a 3×3 matrix, diagonalises its symmetric part, and keeps the eigenvalues (the principal values) and the eigenvectors (the principal axes, one per column). Its `order` attribute sets which eigenvalue comes first, and the eigenvectors follow the same order.

### 2.1 - Building a tensor from a matrix

We use the ¹H shielding tensor of atom 4 in the ethanol file, in ppm. The matrix is taken directly from the `ms` array of the `Atoms` object.

In [2]:
ethanol = ase_io.read('tutorial_data/ethanol.magres')

sigma = ethanol.arrays['ms'][4]
t = NMRTensor(sigma)

sym = (sigma + sigma.T) / 2
E = t.eigenvectors
print('Eigenvalues (ppm):', np.round(t.eigenvalues, 2))
print('Eigenvectors (columns are the principal axes):')
print(np.round(E, 3))
print('E diag(eigenvalues) E^T rebuilds the symmetric part:',
      bool(np.allclose(E @ np.diag(t.eigenvalues) @ E.T, sym)))

Eigenvalues (ppm): [22.64 27.56 31.97]
Eigenvectors (columns are the principal axes):
[[ 0.759 -0.382 -0.527]
 [ 0.161 -0.674  0.721]
 [-0.63  -0.632 -0.451]]
E diag(eigenvalues) E^T rebuilds the symmetric part: True


Because the raw shielding matrix is not strictly symmetric, `NMRTensor` diagonalises its symmetric part, $(\sigma + \sigma^\mathsf{T})/2$, and discards the antisymmetric part. Eigenvectors are defined only up to sign; neither point affects the principal values or the reconstruction above.

### 2.2 - Eigenvalue ordering conventions

Soprano's `order` argument selects one of four eigenvalue orderings, where $\lambda_1, \lambda_2, \lambda_3$ are the eigenvalues in returned order and $\lambda_{\text{iso}} = (\lambda_1 + \lambda_2 + \lambda_3)/3$ is the isotropic average:

- `'i'` (increasing): $\lambda_1 \le \lambda_2 \le \lambda_3$ by algebraic value. Default for a bare `NMRTensor`.
- `'d'` (decreasing): $\lambda_1 \ge \lambda_2 \ge \lambda_3$ by algebraic value.
- `'h'` (Haeberlen): $|\lambda_3 - \lambda_{\text{iso}}| \ge |\lambda_1 - \lambda_{\text{iso}}| \ge |\lambda_2 - \lambda_{\text{iso}}|$, i.e. $|\sigma_{zz} - \sigma_{\text{iso}}| \ge |\sigma_{xx} - \sigma_{\text{iso}}| \ge |\sigma_{yy} - \sigma_{\text{iso}}|$ (so $z$ is furthest from the isotropic value and $y$ is closest). Default for shielding tensors (`MSTensor`, `MSEuler`, and other shielding properties).
- `'n'` (NQR): $|\lambda_1| \le |\lambda_2| \le |\lambda_3|$ by absolute value, i.e. $|V_{xx}| \le |V_{yy}| \le |V_{zz}|$. Default for EFG tensors. Intended for traceless tensors; if called on a tensor with non-zero trace, Soprano warns and still sorts by magnitude, so we do not use `'n'` for shielding tensors here.

For a traceless EFG tensor, the Haeberlen order coincides with $|V_{yy}| \le |V_{xx}| \le |V_{zz}|$ (equivalently $|V_{zz}| \ge |V_{xx}| \ge |V_{yy}|$), which is what SIMPSON expects (Section 4.3).

The table below shows the same ethanol shielding tensor in the three orderings suited to shielding.

In [3]:
rows = {}
for order in ['i', 'd', 'h']:
    t.order = order
    rows[f"'{order}'"] = t.eigenvalues
pd.DataFrame(rows, index=['λ₁', 'λ₂', 'λ₃']).T.round(2)

,λ₁,λ₂,λ₃
'i',22.64,27.56,31.97
'd',31.97,27.56,22.64
'h',31.97,27.56,22.64


The three principal values are identical in every row; only their assignment to $(x, y, z)$ changes. For atom 4, the component furthest from $\sigma_{\text{iso}} \approx 27.4\text{ ppm}$ is the smallest value ($22.64\text{ ppm}$), so Haeberlen ordering places it last (matching decreasing order for this particular site, though in general Haeberlen depends on deviation from $\sigma_{\text{iso}}$ rather than algebraic value). Because permuting the eigenvalues permutes the eigenvector columns, the choice of ordering directly changes the Euler angles.

## 3 - Euler angle conventions

Three Euler angles $(\alpha, \beta, \gamma)$ specify the rotation relating the common crystal/molecular frame to a tensor's principal-axis system (PAS). Three convention choices affect the numerical values: the rotation sequence (ZYZ vs ZXZ), the rotation sense (active vs passive), and the eigenvalue ordering (Section 2.2). Different simulation codes make different choices, so a triple of Euler angles is ambiguous unless all three conventions are fixed.

### 3.1 - ZYZ vs ZXZ, active vs passive

ZYZ applies rotations about $z$, then $y$, then $z$; ZXZ uses $x$ for the middle rotation. ZYZ is standard in NMR and is the default in both Soprano and SIMPSON; ZXZ is also supported to match TensorView.

Active and passive Euler angles parametrise inverse rotations of one another (following TensorView's convention; see [SIMPSON tensor conventions](../simpson-conventions)). Let $E$ be the eigenvector matrix whose columns are the PAS axes expressed in the common crystal/molecular frame, so that $T = E\,\mathrm{diag}(\lambda)\,E^\mathsf{T}$. With **active** ZYZ angles,

$$
E = R_z(\alpha)\,R_y(\beta)\,R_z(\gamma),
$$

whereas with **passive** ZYZ angles,

$$
E = R_z(-\gamma)\,R_y(-\beta)\,R_z(-\alpha).
$$

Because active and passive angles produce different numbers for the same tensor, passing active angles to a simulator that expects passive ones silently rotates the tensor to the wrong orientation. SIMPSON uses **passive ZYZ** angles in degrees. MRSimulator's implementation uses a distinct parametrisation, $E = R_z(\gamma)\,R_y(\beta)\,R_z(\alpha)$ in radians (Section 7.2 and [SIMPSON tensor conventions](../simpson-conventions)). Soprano converts to each program's convention automatically on export.

Below we reconstruct the tensor from all four combinations to confirm that each describes the exact same physical tensor.

In [4]:
def rot(axis, angle):
    return R.from_euler(axis, angle).as_matrix()


def rebuild(angles, convention, passive):
    a, b, g = angles
    middle = 'y' if convention == 'zyz' else 'x'
    if passive:
        E_rec = rot('z', -g) @ rot(middle, -b) @ rot('z', -a)
    else:
        E_rec = rot('z', a) @ rot(middle, b) @ rot('z', g)
    return E_rec @ np.diag(t.eigenvalues) @ E_rec.T


t.order = 'h'
rows = []
for convention in ['zyz', 'zxz']:
    for passive in [False, True]:
        angles = t.euler_angles(convention, passive=passive)
        rows.append({
            'convention': convention.upper(),
            'sense': 'passive' if passive else 'active',
            'α (°)': np.degrees(angles[0]),
            'β (°)': np.degrees(angles[1]),
            'γ (°)': np.degrees(angles[2]),
            'rebuilds tensor': bool(np.allclose(rebuild(angles, convention, passive), sym)),
        })
pd.DataFrame(rows).round(2)

,convention,sense,α (°),β (°),γ (°),rebuilds tensor
0,ZYZ,active,191.98,50.93,125.47,True
1,ZYZ,passive,54.53,50.93,348.02,True
2,ZXZ,active,281.98,50.93,35.47,True
3,ZXZ,passive,144.53,50.93,258.02,True


All four sets rebuild the tensor, but they are four different sets of numbers. The passive ZYZ row is the one SIMPSON uses (section 4.3).

### 3.2 - Equivalent Euler angle sets

Euler angles for a second-rank tensor are not unique. A $180^\circ$ half-turn about any principal axis maps each axis onto itself or its negative, leaving the diagonal tensor invariant. Together with the identity, these four rotations produce **4 equivalent angle sets** for a single tensor (for a fixed convention, sense, and ordering).

For the *relative* orientation between two tensors, each tensor contributes this 4-fold symmetry, yielding **16 equivalent sets** ($4 \times 4$, shown in Section 4.4).

`equivalent_euler_angles` returns all four sets for a tensor. `euler_angles` returns one canonical representative chosen with $\beta \in [0^\circ, 90^\circ]$ (the first row below).

In [5]:
sets = t.equivalent_euler_angles('zyz', passive=False)
pd.DataFrame(
    [
        {
            'α (°)': np.degrees(s[0]),
            'β (°)': np.degrees(s[1]),
            'γ (°)': np.degrees(s[2]),
            'rebuilds tensor': bool(np.allclose(rebuild(s, 'zyz', False), sym)),
        }
        for s in sets
    ],
    index=pd.Index(range(1, len(sets) + 1), name='set'),
).round(2)

,α (°),β (°),γ (°),rebuilds tensor
set,,,,
1,191.98,50.93,125.47,True
2,191.98,50.93,305.47,True
3,11.98,129.07,54.53,True
4,11.98,129.07,234.53,True


Any of the four rows gives the same tensor. When you compare angles from two programs, check whether they agree in one of these sets before concluding that the orientations differ.

## 4 - Orientations from .magres data

A `.magres` file stores each tensor as a $3 \times 3$ Cartesian matrix in the crystal/molecular frame of the structure. Soprano's property classes read these matrices per site and return the Euler angles directly (`MSEuler` and `EFGEuler` return radians; the tables below convert to degrees).

Every table uses the same row labels: the 0-based atom index (matching ASE and Soprano) and element symbol.

In [6]:
atom_labels = [f"{i} {sym_}" for i, sym_ in enumerate(ethanol.get_chemical_symbols())]
ANGLE_COLUMNS = ['α (°)', 'β (°)', 'γ (°)']


def angle_table(eulers, labels):
    return pd.DataFrame(
        np.degrees(eulers), index=pd.Index(labels, name='atom (0-based)'), columns=ANGLE_COLUMNS
    ).round(1)

### 4.1 - Shielding (MSEuler)

`MSEuler` gives the Euler angles of every shielding tensor. Its defaults are Haeberlen ordering and active ZYZ angles. We write the arguments out so that the choices are visible.

In [7]:
ms_eulers = MSEuler.get(ethanol, order='h', convention='zyz', passive=False)
angle_table(ms_eulers, atom_labels)

,α (°),β (°),γ (°)
atom (0-based),,,
0 H,28.0,45.3,26.8
1 H,272.7,77.4,93.2
2 H,182.0,35.1,166.5
3 H,182.7,83.2,115.1
4 H,192.0,50.9,125.5
5 H,299.1,42.3,25.4
6 C,93.1,47.4,153.0
7 C,9.3,54.9,41.4
8 O,183.0,59.5,126.0


Each row describes an individual tensor in the common crystal/molecular frame. Angles from different atoms cannot be compared directly because each refers to its own PAS; the frame-independent quantity is the *relative* orientation between two tensors (Section 4.4).

### 4.2 - EFG (EFGEuler)

EFG tensors are traceless. Their default ordering in Soprano is NQR, which differs from the shielding default (Haeberlen), so the default EFG angles are not directly comparable with the default shielding angles. The eigenvalues show the difference: for the first ¹H, the NQR order and the Haeberlen order differ by swapping the first two entries.

In [8]:
efg_eigenvalues = {order: EFGTensor.get(ethanol, order=order)[0].eigenvalues for order in ['n', 'h']}
pd.DataFrame(efg_eigenvalues, index=['1st', '2nd', '3rd']).T.round(4)

,1st,2nd,3rd
n,-0.1416,-0.1468,0.2884
h,-0.1468,-0.1416,0.2884


Only quadrupolar nuclei, such as ²H or ¹⁷O, have a quadrupolar splitting that depends on the EFG. The EFG of a ¹H or ¹³C is still in the file, and still has an orientation, but it does not affect the spectrum. We list every site below, and when we simulate we keep only the quadrupolar ones.

In [9]:
efg_eulers_default = EFGEuler.get(ethanol)
angle_table(efg_eulers_default, atom_labels)

,α (°),β (°),γ (°)
atom (0-based),,,
0 H,11.0,53.9,124.7
1 H,281.4,58.6,177.2
2 H,193.2,54.8,63.1
3 H,192.6,54.9,1.7
4 H,282.1,58.5,125.3
5 H,283.0,54.6,119.2
6 C,107.2,53.9,32.4
7 C,13.8,54.5,129.7
8 O,146.6,44.4,94.8


### 4.3 - Exporting angles to SIMPSON

SIMPSON requires two settings: **Haeberlen ordering** ($|V_{yy}| \le |V_{xx}| \le |V_{zz}|$ for an EFG) and **passive ZYZ** angles in degrees (see [SIMPSON tensor conventions](../simpson-conventions)). Soprano's `to_simpson()` applies both automatically.

When calling `MSEuler` or `EFGEuler` manually for SIMPSON, always specify `order='h'`, `convention='zyz'`, `passive=True`. The default `EFGEuler` call in Section 4.2 (NQR, active) will not match SIMPSON's angles.

Note why you cannot fix NQR-ordered EFG angles with a single constant $90^\circ$ offset: comparing Haeberlen and NQR active angles below, $\Delta\alpha = 0^\circ$ at every site, while $\Delta\gamma$ flips between $+90^\circ$ and $-90^\circ$ from site to site depending on handedness. Passing `order='h'` handles this cleanly for every site.

In [10]:
eulers_nqr_active = EFGEuler.get(ethanol, order='n', convention='zyz', passive=False)
eulers_h_active = EFGEuler.get(ethanol, order='h', convention='zyz', passive=False)


def wrap(angle):
    return (angle + np.pi) % (2 * np.pi) - np.pi


difference = pd.DataFrame(
    {
        'Δα (°)': np.degrees(wrap(eulers_h_active[:, 0] - eulers_nqr_active[:, 0])),
        'Δγ (°)': np.degrees(wrap(eulers_h_active[:, 2] - eulers_nqr_active[:, 2])),
    },
    index=pd.Index(atom_labels, name='atom (0-based)'),
).round(1) + 0.0
difference

,Δα (°),Δγ (°)
atom (0-based),,
0 H,0.0,-90.0
1 H,0.0,-90.0
2 H,0.0,90.0
3 H,0.0,90.0
4 H,0.0,-90.0
5 H,0.0,-90.0
6 C,0.0,90.0
7 C,0.0,-90.0
8 O,0.0,-90.0


For every site in ethanol, $\Delta\alpha = 0^\circ$ and $\Delta\gamma = \pm 90^\circ$, with the sign changing from site to site.

Next we print the SIMPSON-convention Euler angles for the methyl group (selecting its four atoms with a distance sphere, covered in Section 5.2).

In [11]:
c1 = AtomSelection.from_selection_string(ethanol, 'C.1').indices[0]
methyl_sel = AtomSelection.from_sphere(ethanol, ethanol.positions[c1], 1.5, periodic=True)
methyl_idx = list(methyl_sel.indices)
methyl_labels = [atom_labels[i] for i in methyl_idx]

ms_simpson = MSEuler.get(ethanol, order='h', convention='zyz', passive=True)
efg_simpson = EFGEuler.get(ethanol, order='h', convention='zyz', passive=True)

pd.concat(
    {
        'shielding (MSEuler, SIMPSON)': angle_table(ms_simpson[methyl_idx], methyl_labels),
        'EFG (EFGEuler, SIMPSON)': angle_table(efg_simpson[methyl_idx], methyl_labels),
    },
    axis=1,
)

shielding (MSEuler, SIMPSON)               \
                                      α (°) β (°)  γ (°)   
atom (0-based)                                             
0 H                                   153.2  45.3  152.0   
1 H                                    86.8  77.4  267.3   
2 H                                    13.5  35.1  358.0   
6 C                                    27.0  47.4   86.9   

               EFG (EFGEuler, SIMPSON)               
                                 α (°) β (°)  γ (°)  
atom (0-based)                                       
0 H                              145.3  53.9  169.0  
1 H                               92.8  58.6  258.6  
2 H                               26.9  54.8  346.8  
6 C                               57.6  53.9   72.8

These are the $(\alpha, \beta, \gamma)$ triples that Soprano writes on the `shift` and `quadrupole` lines of a SIMPSON `spinsys` block. Section 8 verifies the EFG values directly against the exported lines.

### 4.4 - Relative orientations

Individual tensor Euler angles depend on the arbitrary crystal/molecular frame, whereas the *relative* orientation between two tensors is frame-independent—and is what determines multi-interaction powder lineshapes.

`A.euler_to(B)` returns the Euler angles of tensor `B` in the PAS of tensor `A` (the rotation $E_A^\mathsf{T} E_B$). Always use the same eigenvalue ordering (here Haeberlen) for both tensors so conventions are not mixed.

First, we compute the shielding-to-EFG relative orientation at each site (EFG PAS expressed in the shielding PAS).

In [12]:
ms_h = MSTensor.get(ethanol, order='h')
efg_h = EFGTensor.get(ethanol, order='h')

ms_to_efg = np.array([ms.euler_to(efg) for ms, efg in zip(ms_h, efg_h)])
angle_table(ms_to_efg, atom_labels)

,α (°),β (°),γ (°)
atom (0-based),,,
0 H,89.1,15.4,94.1
1 H,65.0,20.4,112.3
2 H,140.5,21.1,44.9
3 H,311.6,29.8,68.1
4 H,170.1,70.8,160.0
5 H,75.2,17.1,111.7
6 C,89.1,12.6,112.5
7 C,123.6,3.7,55.4
8 O,285.5,32.1,110.2


For a quadrupolar site, the relative orientation of the EFG and the shielding tensor matters for the `quadrupole_x_shift` cross-term in section 8.

For two shielding tensors, `euler_to` gives one relative rotation, and `equivalent_euler_to` gives all 16 equivalent sets. We take the two carbons, atoms 6 and 7.

In [13]:
single = np.degrees(ms_h[6].euler_to(ms_h[7]))
print('euler_to (one representative), degrees:', np.round(single, 2))

relative_sets = ms_h[6].equivalent_euler_to(ms_h[7])
pd.DataFrame(
    np.degrees(relative_sets), columns=ANGLE_COLUMNS,
    index=pd.Index(range(1, len(relative_sets) + 1), name='set'),
).round(2)

euler_to (one representative), degrees: [267.05  63.05  83.41]


,α (°),β (°),γ (°)
set,,,
1,267.05,63.05,83.41
2,87.05,116.95,276.59
3,87.05,116.95,96.59
4,267.05,63.05,263.41
5,92.95,116.95,263.41
6,272.95,63.05,96.59
7,272.95,63.05,276.59
8,92.95,116.95,83.41
9,272.95,116.95,263.41


The 16 rows describe the same relative rotation. `euler_to` returns one of them, normalised in the same way as `euler_angles` (here it is the first row). When you compare with another program, the set to match is the one that agrees with that program's own choice of ranges and tensor frames, so compare the rotation rather than the numbers.

### 4.5 - Dipolar tensor orientation

Dipolar tensors are not stored in a `.magres` file. Soprano builds them from the geometry and the gyromagnetic ratios. For a pair of nuclei the tensor is axially symmetric, with its unique axis along the internuclear vector. That gives a useful check: the unique eigenvector of the tensor should coincide with the bond.

We use the directly bonded H and C pair, atoms 0 and 6 (0-based) in ethanol.

In [14]:
pair = (0, 6)
i, j = pair
symbols = ethanol.get_chemical_symbols()

D = DipolarTensor().get(ethanol)[pair]
bond = ethanol.get_distance(i, j, mic=True, vector=True)
bond_unit = bond / np.linalg.norm(bond)
unique_axis = D.eigenvectors[:, 2]

dip_euler = DipolarEuler().get(ethanol, convention='zyz', passive=True, degrees=True)[pair]
theta = np.degrees(np.arccos(bond_unit[2]))
phi = np.degrees(np.arctan2(bond_unit[1], bond_unit[0]))

pd.DataFrame(
    {
        'value': [
            f"{i} {symbols[i]}",
            f"{j} {symbols[j]}",
            round(float(ethanol.get_distance(i, j, mic=True)), 3),
            round(float(dip_euler[0]), 2),
            round(float(dip_euler[1]), 2),
            round(float(dip_euler[2]), 2),
            round(float(theta), 2),
            round(float(phi), 2),
            bool(np.isclose(abs(bond_unit @ unique_axis), 1.0)),
        ]
    },
    index=[
        'atom i (0-based)', 'atom j (0-based)', 'distance (Å)',
        'α, passive ZYZ (°)', 'β, passive ZYZ (°)', 'γ, passive ZYZ (°)',
        'bond polar angle θ (°)', 'bond azimuth φ (°)',
        'unique axis is along the bond',
    ],
)

,value
atom i (0-based),0 H
atom j (0-based),6 C
distance (Å),1.094
"α, passive ZYZ (°)",0.0
"β, passive ZYZ (°)",53.82
"γ, passive ZYZ (°)",168.75
bond polar angle θ (°),53.82
bond azimuth φ (°),11.25
unique axis is along the bond,True


For an axially symmetric tensor one Euler angle is free and Soprano sets it to zero—in the passive convention used here, that is $\alpha = 0^\circ$. The remaining two angles specify the unique axis: $\beta$ equals the bond's polar angle $\theta$, and $180^\circ - \gamma$ equals its azimuth $\phi$ in the crystal/molecular frame, matching [SIMPSON tensor conventions](../simpson-conventions). The unique-axis dot product confirms that the tensor aligns with the C–H bond.

## 5 - Selecting the spins to simulate

A spin system only needs the spins that matter for the simulation. Selections are made with `AtomSelection`, as in [Tutorial 3](./03-atomselection_transforms.ipynb). We show three ways to choose spins: by element, by label or index, and by distance. The selections are passed to the functions in section 6.

### 5.1 - By element, label, or index

`from_element` selects every atom of one element. `from_selection_string` takes a comma-separated list. A bare symbol such as `C` selects every carbon, `C.1` means the first carbon in the file (the count is 1-based within the element, not the atom index), and `N1` selects the atom with the label `N1` in the magres file.

In [15]:
selections = {
    'H': AtomSelection.from_element(ethanol, 'H'),
    'C,O': AtomSelection.from_selection_string(ethanol, 'C,O'),
    'C.1': AtomSelection.from_selection_string(ethanol, 'C.1'),
}
pd.DataFrame(
    {
        'selection': list(selections.keys()),
        'number of atoms': [len(s) for s in selections.values()],
        'atom indices (0-based)': [str(list(map(int, s.indices))) for s in selections.values()],
    }
).set_index('selection')

,number of atoms,atom indices (0-based)
selection,,
H,6,"[0, 1, 2, 3, 4, 5]"
"C,O",3,"[6, 7, 8]"
C.1,1,[6]


Labels are especially useful in molecular crystals, where a label selection matches every symmetry copy in the unit cell. Below we load `EDIZUM.magres` (four molecules per cell) and call `reload_as_molecular_crystal` so that molecules straddling periodic boundaries are reconstructed intact before we inspect and select atoms.

In [16]:
EDIZUM = ase_io.read('tutorial_data/EDIZUM.magres')
EDIZUM = reload_as_molecular_crystal(EDIZUM)

sel_labels = AtomSelection.from_selection_string(EDIZUM, 'N1,C1,C6,C7')
print(f"Atoms selected by 'N1,C1,C6,C7': {len(sel_labels)}")

Atoms selected by 'N1,C1,C6,C7': 16


The selection picks four atoms in each of the four molecules—sixteen atoms in total—so each label appears four times. We visualise a selection in 3D in Section 5.2.

In [17]:
sel_idx = sel_labels.indices
pd.DataFrame({
    'index (0-based)': sel_idx,
    'label': EDIZUM.get_array('labels')[sel_idx],
    'element': np.array(EDIZUM.get_chemical_symbols())[sel_idx],
}).set_index('index (0-based)')

,label,element
index (0-based),,
136,N1,N
137,N1,N
138,N1,N
139,N1,N
76,C1,C
77,C1,C
78,C1,C
79,C1,C
96,C6,C


### 5.2 - By distance

`from_sphere` selects all atoms within a given radius of a point. Setting `periodic=True` accounts for periodic images, which matters whenever a molecule crosses a unit-cell boundary. Below we select the ethanol methyl group—`C.1` and every atom within 1.5 Å of it—which we use to build the spin system in Section 6.

In [18]:
c1 = AtomSelection.from_selection_string(ethanol, 'C.1').indices[0]
methyl_sel = AtomSelection.from_sphere(ethanol, ethanol.positions[c1], 1.5, periodic=True)
methyl_idx = list(methyl_sel.indices)
methyl = methyl_sel.subset(ethanol)

pd.DataFrame(
    {
        'index (0-based)': methyl_idx,
        'element': [symbols[k] for k in methyl_idx],
    }
).set_index('index (0-based)')

,element
index (0-based),
0,H
1,H
2,H
6,C


The selection is based on distance, so it would pick up any other atom inside the sphere. For a single molecule with a clear centre this is a convenient way to get a local group. A sphere can also catch atoms that belong to a neighbouring molecule, so check the selection before you use it on a crystal.

The sphere below shows the same kind of selection in the EDIZUM crystal, around the N atom with index 138. The atoms inside the sphere are the selection, and the code prints how many there are.

In [19]:
def structure_only(atoms):
    # Strip extra .magres arrays for lighter 3D rendering in WeasWidget
    return Atoms(symbols=atoms.get_chemical_symbols(), positions=atoms.positions, cell=atoms.cell, pbc=atoms.pbc)


centre_index = 138
sphere_centre = EDIZUM.positions[centre_index]
sphere_radius = 2.0
sel_sphere = AtomSelection.from_sphere(EDIZUM, sphere_centre, sphere_radius)
print(f"Atoms within {sphere_radius} Å of atom {centre_index}: {len(sel_sphere)}")

v_full = WeasWidget()
v_full.from_ase(structure_only(EDIZUM))
v_full.imp.settings = [{
    "type": "sphere",
    "opacity": 0.5,
    "shape": {"radius": sphere_radius},
    "instances": [{"position": sphere_centre, "color": "#fff70c"}],
}]

widgets.VBox([
    widgets.HTML(f"<b>Whole cell; the sphere marks the atoms within {sphere_radius:.1f} Å of atom {centre_index}</b>"),
    v_full,
])

Atoms within 2.0 Å of atom 138: 4


## 6 - Building a spin system

A `SpinSystem` is a list of `Site` objects, one per spin with its shielding and EFG, and a list of `Coupling` objects, which are dipolar or J couplings between pairs of spins. The helper functions take an `ase.Atoms` object and a selection and build these lists. You can also build each piece by hand, which the subsections show.

We use the ethanol methyl group throughout. It has three ¹H and one ¹³C, so the system stays small enough to check by eye.

### 6.1 - Sites

A `Site` represents one nuclear spin with a label, isotope, index, and optional shielding and EFG tensors. `get_sites` builds one site per selected atom using the specified `isotopes`. Shieldings are stored as raw $\sigma$ values; chemical shifts $\delta$ are computed when references are supplied (Section 6.4).

We pass ¹H and ¹³C explicitly rather than relying on defaults.

In [20]:
methyl_sites = get_sites(ethanol, selection=methyl_sel, isotopes={'H': 1, 'C': 13})

pd.DataFrame([
    {
        'index': s.index,
        'isotope': s.isotope,
        'σ_iso (ppm)': s.ms.isotropy,
        'ζ_σ (ppm)': s.ms.reduced_anisotropy,
        'η_σ': s.ms.asymmetry,
    }
    for s in methyl_sites
]).set_index('index').round(2)

,isotope,σ_iso (ppm),ζ_σ (ppm),η_σ
index,,,,
0,1H,29.59,5.96,0.14
1,1H,30.26,5.46,0.21
2,1H,30.10,4.86,0.06
3,13C,156.47,22.53,0.70


Indices here are relative to the methyl subset (0 to 3). Shielding values are raw $\sigma$ (larger $\sigma$ means more shielded); $\zeta_\sigma$ and $\eta_\sigma$ are the reduced anisotropy and asymmetry defined in [Tutorial 5a](./05a-nmr-parameters.ipynb), Section 3.4.

A `Site` can also be constructed manually when tensors come from another source. Because ¹H has spin $\tfrac{1}{2}$ (zero quadrupole moment), we omit `efg`.

In [21]:
sigma_H = np.array([
    [30.29817962,  1.20510693,  3.67274493],
    [ 1.96313295, 27.57652505,  2.57545224],
    [ 4.21834132,  2.16271308, 30.90315252]])

site_H = Site(
    label='H1a',
    isotope='1H',
    index=0,
    ms=MagneticShielding(sigma_H, species='1H'),
)
print(f"Label: {site_H.label}, isotope: {site_H.isotope}, index: {site_H.index}")
print(f"σ_iso = {site_H.ms.isotropy:.2f} ppm, EFG present: {site_H.efg is not None}")

site_from_dict = Site.from_dict({
    'label': 'H1a',
    'isotope': '1H',
    'index': 0,
    'ms': {'data': sigma_H, 'species': '1H'},
})
print('Same site from a dictionary:', bool(site_H == site_from_dict))

Label: H1a, isotope: 1H, index: 0
σ_iso = 29.59 ppm, EFG present: False
Same site from a dictionary: True


### 6.2 - Dipolar couplings

`get_dipolar_couplings` computes one `DipolarCoupling` per spin pair for the chosen isotopes. By default it includes minimum-image pairs across the whole unit cell—both intramolecular and intermolecular. To restrict a crystal to intramolecular couplings, split it into individual molecules with `Molecules` first.

We start with all pairs in the ethanol methyl group (¹H and ¹³C).

In [22]:
dipolar = get_dipolar_couplings(methyl, isotopes={'H': 1, 'C': 13})

pd.DataFrame([
    {
        'site i': f"{d.site_i} ({d.species1})",
        'site j': f"{d.site_j} ({d.species2})",
        'd (Hz)': d.coupling_constant,
    }
    for d in dipolar
]).round(0)

,site i,site j,d (Hz)
0,0 (1H),1 (1H),-21681.0
1,1 (1H),2 (1H),-21554.0
2,0 (1H),3 (13C),-23051.0
3,2 (1H),3 (13C),-23188.0
4,0 (1H),2 (1H),-21358.0
5,1 (1H),3 (13C),-23005.0


The methyl group contains three ¹H–¹H pairs and three ¹H–¹³C pairs. A rotating methyl group averages its internal dipolar couplings, and a static spin system like this one does not include that averaging. The [Dipolar averaging](../dipolar-averaging) page explains how Soprano handles the averaged case.

Next, a single molecule in the EDIZUM crystal. We take its first molecule and select the H14 and C9 pair, which is intramolecular.

In [23]:
molecules = Molecules.get(EDIZUM)
mol0 = molecules[0].subset(EDIZUM, use_cell_indices=True)
sel_H14 = AtomSelection.from_selection_string(mol0, 'H14')
sel_C9 = AtomSelection.from_selection_string(mol0, 'C9')

mol_dipolar = get_dipolar_couplings(mol0, sel_i=sel_H14, sel_j=sel_C9, isotopes={'H': 1, 'C': 13})

pd.DataFrame([
    {
        'label i': mol0.arrays['labels'][d.site_i],
        'label j': mol0.arrays['labels'][d.site_j],
        'd (Hz)': d.coupling_constant,
    }
    for d in mol_dipolar
]).round(0)

,label i,label j,d (Hz)
0,C9,H14,-32133.0


Because indices in `mol0` are local to that single molecule, CIF labels are a safer way to identify atoms than raw indices.

A `DipolarCoupling` can also be built by hand. Below we use the bonded ¹H–¹³C coupling constant from the methyl group ($-23050.6\text{ Hz}$) and place its unique axis along $z$ of the common crystal/molecular frame. Because the tensor is already diagonal in that frame, all three Euler angles are zero.

In [24]:
d_CH = -23050.6  # Hz, the bonded 1H-13C value from the methyl group above

dip_manual = DipolarCoupling(
    site_i=0,
    site_j=1,
    species1='1H',
    species2='13C',
    tensor=NMRTensor(np.diag([-d_CH, -d_CH, 2 * d_CH]), order='n'),
    tag='manual',
)
print(f"Coupling constant: {dip_manual.coupling_constant:.1f} Hz")
print(dip_manual.to_simpson())

Coupling constant: -23050.6 Hz
dipole 1 2 -23050.600000 0.000000 0.000000 0.000000


The coupling constant is recovered cleanly from the tensor (whose eigenvalues are $-d, -d, 2d$, so $d$ is half the reduced anisotropy). Because the PAS coincides with the common crystal/molecular frame, the exported SIMPSON line has zero Euler angles; rotating the tensor would change those angles while leaving the coupling constant unchanged.

### 6.3 - J couplings

`get_j_couplings` works in the same way for indirect spin–spin ($J$) tensors. The `.magres` file stores the reduced coupling tensor $K$ (in $10^{19}\text{ T}^2\text{ J}^{-1}$), which Soprano converts to Hz using the gyromagnetic ratios $\gamma_i, \gamma_j$ of the chosen isotopes:

$$
J = 10^{19}\,\frac{h\,\gamma_i\,\gamma_j}{4\pi^2}\,K.
$$

Below we extract the ¹H–¹³C J couplings between the methyl hydrogens and `C.1`.

In [25]:
j_methyl = get_j_couplings(
    ethanol,
    sel_i=methyl_sel,
    sel_j=AtomSelection.from_selection_string(ethanol, 'C.1'),
    isotopes={'H': 1, 'C': 13},
)

pd.DataFrame([
    {
        'site i': f"{j.site_i} ({j.species1})",
        'site j': f"{j.site_j} ({j.species2})",
        'J (Hz)': j.coupling_constant,
    }
    for j in j_methyl
]).round(2)

,site i,site j,J (Hz)
0,0 (1H),6 (13C),102.11
1,1 (1H),6 (13C),104.58
2,2 (1H),6 (13C),106.02


The three one-bond couplings lie between 102 and 106 Hz—slightly below typical solution values (~125 Hz for an $sp^3$ C–H bond), as discussed in [Tutorial 5a](./05a-nmr-parameters.ipynb), Section 5.2.

To construct a `JCoupling` manually, we enter the reduced $K$ tensor for the H0–C6 pair (rounded to three decimal places) and verify that it reproduces the value above.

In [26]:
K_CH = np.array([
    [31.748, -0.990, -3.425],
    [-1.003, 36.048, -0.722],
    [-3.499, -0.737, 33.599],
])  # reduced K for atoms 0 (1H) and 6 (13C), in 10^19 T^2 J^-1

j_manual = JCoupling(
    site_i=0,
    site_j=6,
    species1='1H',
    species2='13C',
    tensor=NMRTensor(K_CH),
    tag='manual',
)
print(f"J (typed-in K): {j_manual.Jisotropy:.2f} Hz")
print(f"J (from the file): {j_methyl[0].Jisotropy:.2f} Hz")

J (typed-in K): 102.11 Hz
J (from the file): 102.11 Hz


The manual calculation reproduces the file value to within rounding, making it a straightforward way to import J tensors from external sources.

### 6.4 - SpinSystem

`get_spin_system` combines sites and couplings into a `SpinSystem`. Converting magnetic shieldings $\sigma$ to chemical shifts $\delta$ requires `references`; here we pass the illustrative GIPAW/PBE-like values `{'H': 31.0, 'C': 170.0}` for the two elements present (see [Chemical shift referencing](../chemical-shift-referencing) for the full formula and linear calibrations). Note that converting from shielding ($\sigma$) to shift ($\delta$) also flips the sign of the reduced anisotropy ($\zeta_\delta = -\zeta_\sigma$), which you will see on the SIMPSON `shift` lines in Section 7.1.

In [27]:
references = {'H': 31.0, 'C': 170.0}
spin_system = get_spin_system(
    methyl,
    isotopes={'H': 1, 'C': 13},
    references=references,
    include_dipolar=True,
    include_j=True,
)

pd.DataFrame([
    {
        'index': s.index,
        'isotope': s.isotope,
        'σ_iso (ppm)': s.ms.isotropy,
        'δ_iso (ppm)': s.ms.shift,
    }
    for s in spin_system.sites
]).set_index('index').round(2)

,isotope,σ_iso (ppm),δ_iso (ppm)
index,,,
0,1H,29.59,1.41
1,1H,30.26,0.74
2,1H,30.10,0.90
3,13C,156.47,13.54


The shifts $\delta_{\text{iso}}$ differ from $\sigma_{\text{ref}} - \sigma$ only by the small denominator $1 - \sigma_{\text{ref}} \times 10^{-6}$. The methyl protons sit between 0.7 and 1.4 ppm and the carbon at 13.5 ppm—in the expected region for an ethanol CH₃ group given our illustrative references.

The spin system contains all pairwise couplings: six dipolar couplings and six J couplings.

In [28]:
pd.DataFrame([
    {
        'type': 'dipolar' if c.type == 'D' else 'J',
        'site i': c.site_i,
        'site j': c.site_j,
        'coupling (Hz)': c.coupling_constant,
    }
    for c in spin_system.couplings
]).round(1)

,type,site i,site j,coupling (Hz)
0,dipolar,0,1,-21680.8
1,dipolar,1,2,-21554.1
2,dipolar,0,3,-23050.6
3,dipolar,2,3,-23188.4
4,dipolar,0,2,-21358.4
5,dipolar,1,3,-23004.9
6,J,0,1,-5.7
7,J,0,2,-4.7
8,J,0,3,102.1
9,J,1,2,-3.9


## 7 - Exporting the spin system

A `SpinSystem` can be exported as a SIMPSON `spinsys` block or as an MRSimulator dictionary/JSON file. Both export paths apply Haeberlen ordering and each program's Euler-angle convention internally. The exported files are written to a temporary output folder (`tutorial_data/_tmp_output/`).

In [29]:
out_dir = os.path.join('tutorial_data', '_tmp_output')
os.makedirs(out_dir, exist_ok=True)

### 7.1 - SIMPSON

`to_simpson()` generates a `spinsys` block with the observed nucleus listed first in `channels` (here ¹H). `shift` lines use ppm with a `p` suffix, `dipole` and `jcoupling` lines use Hz, and every interaction line ends with passive ZYZ Euler angles $(\alpha, \beta, \gamma)$ in degrees. SIMPSON numbers spins starting from 1 (shifted by $+1$ relative to Soprano's 0-based indices).

In [30]:
spinsys_text = spin_system.to_simpson(observed_nucleus='1H')
with open(os.path.join(out_dir, 'ethanol_methyl.spinsys'), 'w') as f:
    f.write(spinsys_text)
print(spinsys_text)

spinsys {
channels 1H 13C
nuclei 1H 1H 1H 13C
shift 1 1.407424571061708p -5.961194684550957p 0.14196800078113436 153.16339001174384 45.33901509145817 152.03764399252458
shift 2 0.7439720547003713p -5.459174721341341p 0.21082029526274973 86.7747982694457 77.3687949517553 267.33584348090034
shift 3 0.8972743313709369p -4.858548306389421p 0.06045150406398757 13.544414461736181 35.144979662728076 357.95631956940537
shift 4 13.535082781739556p -22.53518881269745p 0.7045565796686414 27.033846028004568 47.430187254621245 86.89587807105443
dipole 1 2 -21680.806313 0.000000 46.871716 215.390219
dipole 2 3 -21554.100943 0.000000 47.866172 301.433844
dipole 1 4 -23050.578304 0.000000 53.817980 168.752350
dipole 3 4 -23188.446630 0.000000 55.042807 346.982604
dipole 1 3 -21358.409400 0.000000 89.346785 167.861592
dipole 2 4 -23004.865808 0.000000 59.023572 258.545627
jcoupling 1 2 -5.669597 -3.689247 0.243029 67.768919 43.352465 28.417002
jcoupling 1 3 -4.695365 -3.711798 0.252916 21.438171 88.627

**Note.** Notice that the CSA reduced anisotropy on the `shift` lines has the opposite sign to $\zeta_\sigma$ in Section 6.1 (e.g. `-5.96p` vs `+5.96 ppm` for site 1) because SIMPSON uses the *shift* sense ($\zeta_\delta = -\zeta_\sigma$). A complete SIMPSON Tcl script also requires `proton_frequency` (e.g. 400 MHz for 9.4 T) in its `par` block to convert `p`-suffixed ppm values to Hz. Full details—including passive ZYZ angles, the $\zeta/2$ scaling of anisotropic J couplings, and dipolar prefactors—are documented in [SIMPSON tensor conventions](../simpson-conventions).

### 7.2 - MRSimulator

`to_mrsimulator()` returns a plain Python dictionary with `sites` and `couplings` lists matching MRSimulator's schema, so it runs without requiring the `mrsimulator` package to be installed. In an environment where MRSimulator is installed, you can load this dictionary (or the exported JSON file) with `mrsimulator.SpinSystem.parse(...)`—check the field names against your installed MRSimulator version before production runs.

Notice two key convention differences relative to SIMPSON: MRSimulator expects Euler angles in **radians** (in its own convention, converted automatically by Soprano), and it uses the **shielding** sign convention for `zeta` (`+5.96 ppm` for the first ¹H here versus `-5.96p` on the SIMPSON `shift` line).

In [31]:
mr_dict = spin_system.to_mrsimulator()
print('First site:')
print(json.dumps(mr_dict['sites'][0], indent=2))
print('First coupling:')
print(json.dumps(mr_dict['couplings'][0], indent=2))

with open(os.path.join(out_dir, 'ethanol_methyl_mrsimulator.json'), 'w') as f:
    f.write(spin_system.to_string(format='mrsimulator'))

First site:
{
  "isotope": "1H",
  "label": "H",
  "isotropic_chemical_shift": 1.407424571061708,
  "shielding_symmetric": {
    "zeta": 5.961194684550957,
    "eta": 0.14196800078113436,
    "alpha": 0.46838720436866677,
    "beta": 0.7913150929573431,
    "gamma": 0.48803517894526216
  }
}
First coupling:
{
  "dipolar": {
    "D": -21680.80631287621,
    "alpha": 0.0,
    "beta": 0.8180657702868758,
    "gamma": 5.665509465554392
  },
  "site_index": [
    0,
    1
  ],
  "isotropic_j": -5.669596850909779,
  "j_symmetric": {
    "zeta": -7.3784935554210165,
    "eta": 0.24302905652337925,
    "alpha": 1.9588018893350831,
    "beta": 0.7566432574831784,
    "gamma": 2.645622399090312
  }
}


## 8 - Advanced: quadrupole cross-terms

*Skippable on a first read.*

To demonstrate quadrupolar export and cross-terms, we substitute ²H (spin $I = 1$) for ¹H to model a hypothetical deuterated methyl group. Spin-$\tfrac{1}{2}$ nuclei (¹H, ¹³C) have no quadrupole moment and emit no quadrupole lines, whereas ²H carries an EFG interaction that can also couple at second order to dipolar and CSA tensors (`quadrupole_x_dipole` and `quadrupole_x_shift` in SIMPSON).

Below we rebuild the methyl spin system with `isotopes={'H': 2, 'C': 13}` and export with ²H as the observed channel.

In [32]:
spin_system_2H = get_spin_system(
    methyl,
    isotopes={'H': 2, 'C': 13},
    references=references,
    include_dipolar=True,
)
text_2H = spin_system_2H.to_simpson(observed_nucleus='2H')
lines_2H = [line for line in text_2H.splitlines() if line.startswith('quadrupole')]
print('\n'.join(lines_2H))

quadrupole 1 2 193809.12541952616 0.01819690682375444 145.33993315700698 53.86885309530375 168.96755681318012
quadrupole 2 2 191322.119193839 0.027063430326907292 92.7562232041228 58.60389462107384 258.62993699661143
quadrupole 3 2 195404.46463165162 0.017008453107052613 26.883489870897588 54.80749090636439 346.82054176593334
quadrupole_x_dipole 1 2
quadrupole_x_dipole 2 1
quadrupole_x_dipole 2 3
quadrupole_x_dipole 3 2
quadrupole_x_dipole 1 4
quadrupole_x_dipole 3 4
quadrupole_x_dipole 1 3
quadrupole_x_dipole 3 1
quadrupole_x_dipole 2 4
quadrupole_x_shift 1
quadrupole_x_shift 2
quadrupole_x_shift 3


The first three lines define the quadrupolar interaction for each ²H site: site index, perturbation order (`2`, meaning both first- and second-order quadrupolar terms), $C_Q$ in Hz, $\eta_Q$, and passive ZYZ Euler angles in degrees. The remaining lines declare the second-order cross-terms: `quadrupole_x_dipole` pairs each ²H quadrupolar tensor with its dipolar couplings, and `quadrupole_x_shift` pairs it with the site's shielding tensor.

The Euler angles on the `quadrupole` lines match the Haeberlen passive `EFGEuler` values from Section 4.3, as verified below.

In [33]:
quad_angles = np.array([[float(x) for x in line.split()[5:8]] for line in lines_2H if line.startswith('quadrupole ')])
print('Angles written match EFGEuler (Haeberlen, passive):',
      bool(np.allclose(quad_angles, np.degrees(efg_simpson[methyl_idx[:3]]))))
print('C_Q (kHz) for the three ²H sites:', [round(float(line.split()[3]) / 1e3, 1) for line in lines_2H if line.startswith('quadrupole ')])

Angles written match EFGEuler (Haeberlen, passive): True
C_Q (kHz) for the three ²H sites: [193.8, 191.3, 195.4]


The $C_Q$ values (~191–195 kHz) lie at the upper end of the typical range for aliphatic C–²H bonds. Because quadrupolar cross-terms enter at second order, their impact on a spectrum depends on $C_Q$, the magnetic field, and the pulse sequence; toggle `include_cross_terms` in `to_simpson()` to test whether they matter for your simulation. As in Section 4.4, it is the *relative* orientation between the EFG and the shielding or dipolar tensor that governs how these cross-terms shape the lineshape.

## 9 - Summary and next steps

In this tutorial you extracted tensor orientations and built export-ready spin systems from `.magres` files:

- Diagonalised tensors with `NMRTensor` and compared Soprano's eigenvalue orderings (`'i'`, `'d'`, `'h'`, `'n'`).
- Verified active vs passive ZYZ/ZXZ Euler-angle conventions, 4-fold single-tensor and 16-fold relative-orientation equivalences, and SIMPSON's Haeberlen + passive ZYZ convention using `MSEuler`, `EFGEuler`, and `DipolarEuler`.
- Selected spins for simulation by element, crystal label, and periodic distance sphere (`AtomSelection`).
- Built `Site`, `DipolarCoupling`, `JCoupling`, and `SpinSystem` objects (both from `.magres` files and manually) and exported them to SIMPSON (`spinsys`) and MRSimulator (dictionary/JSON), including quadrupolar cross-terms for ²H.

**Running the simulations:**

- **SIMPSON:** pass the exported `spinsys` block into a SIMPSON Tcl script (see the [SIMPSON website and manual](https://inano.au.dk/about/research-centers-and-projects/nmr/software/simpson)).
- **SimPYson:** build and run SIMPSON simulations directly from Python (see the [SimPYson repository](https://github.com/nuts-org/simpyson)).
- **MRSimulator:** load the exported dictionary/JSON into MRSimulator (see the [MRSimulator getting started guide](https://mrsimulator.readthedocs.io/en/latest/introduction/getting_started.html)).

For 2D NMR correlation maps (HETCOR and ¹H–¹H DQ/SQ), see [Tutorial 5b](./05b-nmr-2d-spectra.ipynb).